<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="50%"></a>
</p>


<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
}
</style>

# Quaternions and Rotation Representations

Euler angles are easy to read but break at gimbal lock, and the rotation matrix never breaks but needs nine numbers. The flight controller uses a third option, with four numbers and no bad spot: the quaternion. This notebook shows how to read one and how to turn it back into roll, pitch and yaw.

By the end of the notebook you should be able to answer:

- What are the three common ways of writing a rotation, and what is each one good and bad at?

- Why does the ROS 2 IMU message carry a quaternion rather than roll, pitch and yaw?

- How is a quaternion converted into roll, pitch and yaw, and where does gimbal lock come back?

## Quaternions: the practical way out

A hovering Duckiedrone stays far from $\beta = \pm 90^\circ$, so why does gimbal lock matter? The gyroscope never reports an angle, only the noisy angular-velocity measurement $\tilde{\boldsymbol{\omega}}(t)$, so the flight controller finds the orientation by integrating turn rates over time.

If the orientation is kept as Euler angles, angular velocity must first be converted into the rates $\dot{\alpha}$, $\dot{\beta}$, and $\dot{\gamma}$. That conversion uses the inverse of a pitch-dependent matrix and divides by $\cos(\beta)$. Near $\beta = \pm 90^\circ$, $\cos(\beta)$ is close to zero and the result blows up: this is a mathematical singularity. In practice the trouble starts even earlier: at $\beta = \pm 80^\circ$, gyroscope noise is already amplified almost six times on its way into yaw.

A hovering Duckiedrone rarely sees these angles, but the estimator has to work at every orientation, for example after a crash or while the Duckiedrone is turned over by hand. With Euler angles, each of those cases needs special code around the bad spot.

Changing the order of the three angles only moves the bad spot somewhere else. Any description with exactly three numbers has one, like the pole on the sphere. The rotation matrix has no bad spot, but it needs nine numbers that must be kept consistent with each other.

A quaternion has no bad spot either, and needs only four numbers. Using more than three numbers is what avoids the singularity.

### What a quaternion is

Euler angles decompose any orientation into three subsequent rotations. Euler also showed that any orientation can be reached from any other with a single turn, by an angle $\psi$ about a unit axis $\mathbf{u} = (u_x, u_y, u_z)^{\mathsf{T}}$. Its Euclidean length is one, written $\lVert\mathbf{u}\rVert = 1$ ([Euler's rotation theorem](https://en.wikipedia.org/wiki/Euler%27s_rotation_theorem)). A unit quaternion is written $q = (\mathbf{q}_{\mathrm{v}}, q_w)$: unlike the vectors above, it is a scalar-plus-vector quantity. In ROS scalar-last order, its four dimensionless components are $q = (q_x, q_y, q_z, q_w)$. Its vector part is $\mathbf{q}_{\mathrm{v}} = (q_x, q_y, q_z)^{\mathsf{T}}$, and $q_w$ is its scalar part. The axis-angle form is:

$$
q = \left(
\mathbf{u}\,\sin\left(\frac{\psi}{2}\right),\;
\cos\left(\frac{\psi}{2}\right)
\right).
$$

[Figure 1](#figure-1) shows the axis-angle quantities in the quaternion representation.

<a id="figure-1"></a>

<p class="lx-figure" style="width:420px;">
  <img src="../assets/_images/quaternion_axis_angle.png" style="width:100%;" />
</p>

Figure 1: The axis $\mathbf{u}$ and the angle $\psi$ that a quaternion stores.

For a nonzero rotation, $\mathbf{q}_{\mathrm{v}}$ points along the rotation axis and has magnitude $\sin\left(\frac{\psi}{2}\right)$; $q_w = \cos\left(\frac{\psi}{2}\right)$ encodes the angle. Two examples:

- no turn at all, $\psi = 0$: $q = (0, 0, 0, 1)$;

- a $\frac{1}{4}$ turn of yaw, $\psi = 90^\circ$ about
  $\mathbf{u} = (0, 0, 1)^{\mathsf{T}}$:
  $q = (0, 0, \sin(45^\circ), \cos(45^\circ)) = (0, 0, 0.707, 0.707)$.

Why the quaternion uses $\frac{\psi}{2}$ is a longer story, and this video, for example, tells it well: [Quaternions and 3d rotation, explained interactively](https://www.youtube.com/watch?v=zjMuIxRvygQ). For this notebook the definition is enough to read a quaternion.

As for rotation matrices, successive turns can be expressed by multiplying their quaternions, with a multiplication rule of their own. The full algebra of quaternions is beyond the scope of this notebook, and many resources cover it, for example: [Visualizing quaternions](https://eater.net/quaternions) builds it up interactively.

ROS 2 lists $q_w$ last, as this notebook does, but other libraries list it first, so check the order whenever a quaternion passes from one library to another. Mixing up the two orders silently gives a wrong orientation.

### Three properties of quaternions

- __Unit length.__ Only unit quaternions describe a rotation. Therefore,
  $\lVert\mathbf{q}_{\mathrm{v}}\rVert^2 + q_w^2
  = q_x^{2} + q_y^{2} + q_z^{2} + q_w^{2} = 1$.

- __No gimbal lock.__ A small turn always changes the four numbers only a little, whatever the orientation, so nothing goes wrong at $\beta = \pm 90^\circ$, where Euler angles break.

- __Two quaternions for every orientation.__ $q$ and $-q$ describe the same orientation, so two quaternions can look opposite and still mean the same thing. Compare orientations, not numbers.

### Converting to Euler angles

Quaternions have a downside as well: a person cannot read them at a glance, while roll, pitch and yaw are easy to picture. Therefore, quaternions are converted to Euler angles whenever a person or a log file needs to read an orientation. For a normalized quaternion in the ZYX convention of [Equation (4) in Notebook 3](./3-rotation-representations.ipynb#equation-4), with $q = (q_x, q_y, q_z, q_w)$, let $\alpha$, $\beta$, and $\gamma$ denote roll, pitch, and yaw, respectively. Here $\arcsin$ is inverse sine, and $\operatorname{atan2}(y, x)$ uses the signs of both inputs to choose the correct quadrant. The roll angle is:

<span id="equation-1"></span>

$$
\alpha = \operatorname{atan2}\big(2(q_w q_x + q_y q_z),\; 1 - 2(q_x^{2} + q_y^{2})\big).
\tag{1}
$$

The pitch angle is:

<span id="equation-2"></span>

$$
\beta = \arcsin\big(2(q_w q_y - q_z q_x)\big).
\tag{2}
$$

The yaw angle is:

<span id="equation-3"></span>

$$
\gamma = \operatorname{atan2}\big(2(q_w q_z + q_x q_y),\; 1 - 2(q_y^{2} + q_z^{2})\big).
\tag{3}
$$

The helper `quaternion_to_euler` in [imu_math.py](../packages/solution/solution/imu_math.py), which [Notebook 7](./7-interfacing-with-imu.ipynb) uses, implements this.

Gimbal lock comes back here. [Equation (2)](#equation-2) gives the pitch angle $\beta$ through $\arcsin$, whose output is limited to $-90^\circ$ to $+90^\circ$. At $\beta = \pm 90^\circ$, both inputs to $\operatorname{atan2}$ in [Equation (1)](#equation-1) and [Equation (3)](#equation-3) become zero, so roll and yaw can no longer be told apart. This is why the flight controller's estimator keeps orientation as a quaternion and roll, pitch and yaw serve mainly for display. The ROS 2 `sensor_msgs/Imu` message also reports orientation as a quaternion rather than as roll, pitch and yaw. ROS 2 names the four numbers `x`, `y`, `z` and `w`:

```text
geometry_msgs/Quaternion orientation
    float64 x
    float64 y
    float64 z
    float64 w
```

## Comparing rotation representations

[Table 1](#table-1) compares the three ways of representing orientation discussed above.

<a id="table-1"></a>

Table 1: Comparison of three common rotation representations.

|  | Numbers | Easy to read | Bad spot | Best for |
| --- | --- | --- | --- | --- |
| Euler angles | $3$ | Yes | Gimbal lock at $\beta = \pm 90^\circ$ | Showing orientation to a person |
| Rotation matrix | $9$ | No | None | Turning a body-frame vector into a world-frame one |
| Quaternion | $4$ | No | None, but $q$ and $-q$ are the same orientation | Storing, estimating and sending orientation |

Any description with only three numbers has a bad spot, which is why the flight controller works with quaternions and conversion to roll, pitch and yaw is left for display.

## Further reading

[Quaternion fundamentals](https://docs.ros.org/en/jazzy/Tutorials/Intermediate/Tf2/Quaternion-Fundamentals.html) shows how quaternions are created and combined in ROS 2 code.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()